# Semana 10 — Gabarito Revisado dos Exercícios de SQL Avançado para Data Warehouse

Este gabarito foi revisado para ficar coerente com os slides finais da Semana 10.

## Ajustes importantes

- `tempo_sk` é uma chave substituta, não uma data.
- Para filtros de período, use `dim_tempo.data`, `dim_tempo.ano`, `dim_tempo.mes` ou `dim_tempo.ano_mes`.
- Para exemplos de índice no DuckDB, use tabelas físicas com sufixo `_base`.
- A Aula 3 foi tratada como teórica: particionamento e materialized views aparecem como interpretação conceitual, não como prática obrigatória em DuckDB.

## 0. Preparação do ambiente

In [ ]:
import duckdb
import pandas as pd

# Ajuste o caminho dos arquivos se necessário.
# Se os CSVs estiverem na mesma pasta do notebook, este código já deve funcionar.

dim_cliente = pd.read_csv('dim_cliente.csv')
dim_produto = pd.read_csv('dim_produto.csv')
dim_tempo = pd.read_csv('dim_tempo.csv')
fato_vendas = pd.read_csv('fato_vendas.csv')

banco_dados = duckdb.connect()

# Registra os DataFrames como tabelas consultáveis pelo DuckDB
banco_dados.register('dim_cliente', dim_cliente)
banco_dados.register('dim_produto', dim_produto)
banco_dados.register('dim_tempo', dim_tempo)
banco_dados.register('fato_vendas', fato_vendas)

# Para a parte de índices, precisamos de tabelas físicas no DuckDB.
# Por isso criamos versões com o sufixo _base.
banco_dados.sql("DROP TABLE IF EXISTS fato_vendas_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_cliente_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_produto_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_tempo_base")

banco_dados.sql("CREATE TABLE fato_vendas_base AS SELECT * FROM fato_vendas")
banco_dados.sql("CREATE TABLE dim_cliente_base AS SELECT * FROM dim_cliente")
banco_dados.sql("CREATE TABLE dim_produto_base AS SELECT * FROM dim_produto")
banco_dados.sql("CREATE TABLE dim_tempo_base AS SELECT * FROM dim_tempo")

---

# Parte 1 — Subconsultas

## Exercício 1 — Vendas acima da média geral

In [ ]:
query = """
SELECT
    venda_sk,
    order_id,
    cliente_sk,
    valor_total
FROM fato_vendas
WHERE valor_total > (
    SELECT AVG(valor_total)
    FROM fato_vendas
)
ORDER BY valor_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 2 — Vendas com frete acima da média geral

In [ ]:
query = """
SELECT
    venda_sk,
    order_id,
    valor_frete,
    valor_total
FROM fato_vendas
WHERE valor_frete > (
    SELECT AVG(valor_frete)
    FROM fato_vendas
)
ORDER BY valor_frete DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 3 — Categorias com quantidade de vendas acima da média

In [ ]:
query = """
SELECT
    categoria,
    qtd_vendas
FROM (
    SELECT
        p.product_category_name AS categoria,
        COUNT(*) AS qtd_vendas
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY p.product_category_name
) categorias
WHERE qtd_vendas > (
    SELECT AVG(qtd_vendas)
    FROM (
        SELECT
            COUNT(*) AS qtd_vendas
        FROM fato_vendas f
        INNER JOIN dim_produto p
            ON f.produto_sk = p.produto_sk
        WHERE p.product_category_name IS NOT NULL
        GROUP BY p.product_category_name
    ) media_categorias
)
ORDER BY qtd_vendas DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 4 — Vendas dos 3 produtos mais vendidos

In [ ]:
query = """
SELECT
    venda_sk,
    order_id,
    produto_sk,
    valor_total
FROM fato_vendas
WHERE produto_sk IN (
    SELECT
        produto_sk
    FROM fato_vendas
    GROUP BY produto_sk
    ORDER BY COUNT(*) DESC
    LIMIT 3
)
ORDER BY produto_sk, valor_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 5 — Estados com receita acima da média dos estados

In [ ]:
query = """
SELECT
    receita_estado.estado,
    ROUND(receita_estado.receita_total, 2) AS receita_total
FROM (
    SELECT
        c.customer_state AS estado,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    GROUP BY c.customer_state
) receita_estado
WHERE receita_estado.receita_total > (
    SELECT AVG(receita_total)
    FROM (
        SELECT
            c.customer_state AS estado,
            SUM(f.valor_total) AS receita_total
        FROM fato_vendas f
        INNER JOIN dim_cliente c
            ON f.cliente_sk = c.cliente_sk
        GROUP BY c.customer_state
    ) media_estado
)
ORDER BY receita_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

---

# Parte 2 — CTEs com WITH

## Exercício 6 — Vendas por estado usando CTE

In [ ]:
query = """
WITH vendas_por_estado AS (
    SELECT
        c.customer_state AS estado,
        COUNT(DISTINCT f.order_id) AS qtd_pedidos,
        COUNT(*) AS qtd_itens_vendidos,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    GROUP BY c.customer_state
)
SELECT
    estado,
    qtd_pedidos,
    qtd_itens_vendidos,
    ROUND(receita_total, 2) AS receita_total
FROM vendas_por_estado
ORDER BY receita_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 7 — Categorias com maior faturamento usando CTE

In [ ]:
query = """
WITH resumo_categorias AS (
    SELECT
        p.product_category_name AS categoria,
        COUNT(*) AS quantidade_vendas,
        SUM(f.valor_total) AS receita_total,
        AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY p.product_category_name
)
SELECT
    categoria,
    quantidade_vendas,
    ROUND(receita_total, 2) AS receita_total,
    ROUND(ticket_medio, 2) AS ticket_medio
FROM resumo_categorias
WHERE receita_total > 100000
ORDER BY receita_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 8 — Receita por estado e categoria usando CTE

In [ ]:
query = """
WITH receita_por_estado_categoria AS (
    SELECT
        c.customer_state AS estado,
        p.product_category_name AS categoria,
        COUNT(*) AS quantidade_vendas,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY
        c.customer_state,
        p.product_category_name
)
SELECT
    estado,
    categoria,
    quantidade_vendas,
    ROUND(receita_total, 2) AS receita_total
FROM receita_por_estado_categoria
WHERE receita_total > 50000
ORDER BY
    estado,
    receita_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 9 — Meses com receita acima da média mensal

In [ ]:
query = """
WITH receita_mensal AS (
    SELECT
        t.ano_mes,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY t.ano_mes
),
media_mensal AS (
    SELECT
        AVG(receita_total) AS media_receita_mensal
    FROM receita_mensal
)
SELECT
    r.ano_mes,
    ROUND(r.receita_total, 2) AS receita_total
FROM receita_mensal r
CROSS JOIN media_mensal m
WHERE r.receita_total > m.media_receita_mensal
ORDER BY r.ano_mes;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 10 — Pipeline de análise por categoria

In [ ]:
query = """
WITH metricas_categoria AS (
    SELECT
        p.product_category_name AS categoria,
        COUNT(*) AS qtd_vendas,
        SUM(f.valor_total) AS receita_total,
        AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY p.product_category_name
)
SELECT
    categoria,
    qtd_vendas,
    ROUND(receita_total, 2) AS receita_total,
    ROUND(ticket_medio, 2) AS ticket_medio
FROM metricas_categoria
WHERE receita_total > 50000
  AND ticket_medio > 100
ORDER BY receita_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

---

# Parte 3 — Window Functions

## Exercício 11 — Ranking das vendas por categoria

In [ ]:
query = """
SELECT
    p.product_category_name AS categoria,
    f.order_id,
    f.valor_total,
    ROW_NUMBER() OVER (
        PARTITION BY p.product_category_name
        ORDER BY f.valor_total DESC
    ) AS ranking_venda_categoria
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
WHERE p.product_category_name IS NOT NULL
ORDER BY
    categoria,
    ranking_venda_categoria;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 12 — Pedido mais caro de cada estado

In [ ]:
query = """
WITH ranking_pedidos_estado AS (
    SELECT
        c.customer_state AS estado,
        f.order_id,
        f.valor_total,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_state
            ORDER BY f.valor_total DESC
        ) AS posicao
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
)
SELECT
    estado,
    order_id,
    ROUND(valor_total, 2) AS valor_total
FROM ranking_pedidos_estado
WHERE posicao = 1
ORDER BY valor_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 13 — Ranking de estados por receita

In [ ]:
query = """
WITH receita_por_estado AS (
    SELECT
        c.customer_state AS estado,
        COUNT(DISTINCT f.order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    GROUP BY c.customer_state
)
SELECT
    estado,
    qtd_pedidos,
    ROUND(receita_total, 2) AS receita_total,
    RANK() OVER (
        ORDER BY receita_total DESC
    ) AS ranking_estado
FROM receita_por_estado
ORDER BY ranking_estado;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 14 — Comparando RANK e DENSE_RANK com empate

In [ ]:
query = """
WITH exemplo_receita_categoria AS (
    SELECT 'beleza_saude' AS categoria, 100000 AS receita_total
    UNION ALL
    SELECT 'cama_mesa_banho', 90000
    UNION ALL
    SELECT 'esporte_lazer', 90000
    UNION ALL
    SELECT 'informatica_acessorios', 70000
    UNION ALL
    SELECT 'moveis_decoracao', 60000
)
SELECT
    categoria,
    receita_total,
    RANK() OVER (
        ORDER BY receita_total DESC
    ) AS rank_categoria,
    DENSE_RANK() OVER (
        ORDER BY receita_total DESC
    ) AS dense_rank_categoria
FROM exemplo_receita_categoria
ORDER BY receita_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 15 — Percentual da venda dentro da categoria

In [ ]:
query = """
SELECT
    p.product_category_name AS categoria,
    f.order_id,
    f.valor_total,
    SUM(f.valor_total) OVER (
        PARTITION BY p.product_category_name
    ) AS total_categoria,
    ROUND(
        100.0 * f.valor_total / SUM(f.valor_total) OVER (
            PARTITION BY p.product_category_name
        ),
        2
    ) AS percentual_na_categoria
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
WHERE p.product_category_name IS NOT NULL
ORDER BY
    categoria,
    percentual_na_categoria DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 16 — Receita acumulada mensal por estado

In [ ]:
query = """
WITH receita_mensal_estado AS (
    SELECT
        c.customer_state AS estado,
        t.ano,
        t.mes,
        t.ano_mes,
        SUM(f.valor_total) AS receita_mes
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY
        c.customer_state,
        t.ano,
        t.mes,
        t.ano_mes
)
SELECT
    estado,
    ano,
    mes,
    ano_mes,
    ROUND(receita_mes, 2) AS receita_mes,
    ROUND(
        SUM(receita_mes) OVER (
            PARTITION BY estado
            ORDER BY ano, mes
        ),
        2
    ) AS receita_acumulada_estado
FROM receita_mensal_estado
ORDER BY
    estado,
    ano,
    mes;
"""

# Para executar:
# banco_dados.sql(query).df()

---

# Parte 4 — Índices e Otimização

## Exercício 17 — Índice para filtro por estado

In [ ]:
query = """
-- Índice sugerido
CREATE INDEX idx_dim_cliente_estado
ON dim_cliente_base (customer_state);

-- Consulta que pode se beneficiar do índice
SELECT
    cliente_sk,
    customer_id,
    customer_state
FROM dim_cliente_base
WHERE customer_state = 'SP';
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 18 — Índice para JOIN e filtro por data

In [ ]:
query = """
-- Índice na chave de junção da tabela fato
CREATE INDEX idx_fato_vendas_tempo_sk
ON fato_vendas_base (tempo_sk);

-- Índice na chave de junção da dimensão tempo
CREATE INDEX idx_dim_tempo_tempo_sk
ON dim_tempo_base (tempo_sk);

-- Índice na coluna usada no filtro de data
CREATE INDEX idx_dim_tempo_data
ON dim_tempo_base (data);

-- Consulta que pode se beneficiar dos índices
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    t.data
FROM fato_vendas_base f
INNER JOIN dim_tempo_base t
    ON f.tempo_sk = t.tempo_sk
WHERE t.data >= '2018-01-01';
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 19 — Índice composto para filtro combinado

In [ ]:
query = """
CREATE INDEX idx_fato_vendas_cliente_produto
ON fato_vendas_base (cliente_sk, produto_sk);

SELECT
    venda_sk,
    order_id,
    cliente_sk,
    produto_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000
  AND produto_sk = 15420;
"""

# Para executar:
# banco_dados.sql(query).df()

---

# Parte 5 — Aula 3 teórica: particionamento e materialized views

## Exercício 20 — Análise conceitual de performance em DW

### Resposta esperada

1. A consulta pode ficar lenta porque precisa ler muitos registros da tabela fato, fazer `JOIN` com várias dimensões e ainda agrupar os dados com `GROUP BY`. Em bases grandes, isso aumenta o custo de leitura, junção e agregação.

2. Particionar a tabela fato por uma coluna de data real, como `data_venda`, ajuda porque o banco pode ler apenas as partes do período consultado. Por exemplo, uma consulta de 2018 não precisaria percorrer todo o histórico.

3. Uma Materialized View com receita por mês, estado e categoria guarda o resultado agregado fisicamente. Assim, o dashboard consulta um resumo pronto, em vez de recalcular todos os `JOINs` e agregações a cada abertura.

4. `tempo_sk` não deve ser tratado como data neste dataset porque é uma chave substituta sequencial. Ela serve para relacionar `fato_vendas` com `dim_tempo`. Para análise temporal, devemos usar `data`, `ano`, `mes` ou `ano_mes`.

---

# Observações para correção

Considere corretas as respostas que:

- usam os nomes reais das tabelas e colunas;
- fazem os `JOINs` pelas chaves corretas;
- usam `product_category_name` ou `product_category_name_english` de forma consistente, quando o arquivo possuir ambas;
- entendem que `tempo_sk` é chave substituta;
- usam CTEs, subconsultas e Window Functions quando o enunciado solicitar;
- nos exercícios de índice, identificam corretamente as colunas do `WHERE` e do `JOIN`.